# Assignment 3: Pre-training, Fine-tuning, and Prompting Strategies for Bug Fixing

### **Course:** GenAI For Software Development (CSCI 455/555) **Instructor:** Prof. Antonio Mastropaolo

## How to Run: 
1. Run section 0 (If needed, restart kernel after section 0 finishes running)
2. Run all cells from top to bottom
3. Training should take many hours
4. Data files automatically get saved to a 'Data' folder <br>
**In my setup and run through, I was able to run up to Section 7 before my GPU ran out of memory. Hence, why there is duplicate code inside of Section 7 -- so that I was able to restart my kernel and start again from Section 7 cleanly, without retraining anything, but Section 0 needed to rerun again.**

**Three pipelines:**
1. **Pipeline A** — Pre-train T5-small on Java code → Fine-tune on bug fixing
2. **Pipeline B** — Fine-tune T5-small from scratch on bug fixing (no pre-training)
3. **Pipeline C** — RAG with Qwen2.5-Coder-1.5B (few-shot) + zero-shot baseline

**Evaluation:** CodeBLEU + Exact Match on the `code_x_glue_cc_code_refinement` test set

---
**Notebook structure:**
- Section 0: Setup & Dependencies
- Section 1: SentencePiece Tokenizer Training
- Section 2: Pre-training — Pipeline A (Span Corruption on 50K Java methods)
- Section 3: Fine-tuning — Pipeline A (pre-trained model → bug fixing)
- Section 4: Fine-tuning — Pipeline B (scratch model → bug fixing)
- Section 5: RAG — Knowledge Base & Retriever
- Section 6: RAG — Inference (few-shot + zero-shot)
- Section 7: Evaluation (CodeBLEU + Exact Match for all 4 configs)
- Section 8: Results Summary

---
# Section 0 — Setup & Dependencies

In [1]:
from platform import python_version
print(python_version())

3.12.6


In [2]:
# Install all required packages
# Run this cell once at the start of your session
%pip install Cmake transformers==4.46.0 tokenizers==0.20.3 sentencepiece datasets torch tqdm codebleu faiss-cpu gensim

print('Dependencies installed.')

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Dependencies installed.


In [1]:
import os
import json
import random
import pickle
import warnings
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

# HuggingFace
from datasets import load_dataset
from transformers import (
    T5Config,
    T5ForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerFast,
)
import sentencepiece as spm

C:\Users\hewas\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Output paths ─────────────────────
BASE_DIR = 'data/'

TOKENIZER_DIR   = f'{BASE_DIR}/tokenizer'
PRETRAIN_DIR    = f'{BASE_DIR}/model_pretrained'
FINETUNE_A_DIR  = f'{BASE_DIR}/model_finetuned_A'
FINETUNE_B_DIR  = f'{BASE_DIR}/model_finetuned_B'
RAG_KB_DIR      = f'{BASE_DIR}/rag_kb'
RESULTS_DIR     = f'{BASE_DIR}/results'

for d in [TOKENIZER_DIR, PRETRAIN_DIR, FINETUNE_A_DIR, FINETUNE_B_DIR, RAG_KB_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Paths ready.')

Device: cuda
Paths ready.


---
# Section 1 — SentencePiece Tokenizer Training

**Requirements:**
- `model_type='unigram'`, `vocab_size=16384`
- Must include T5 special tokens: `<pad>`, `</s>`, `<unk>`, and sentinels `<extra_id_0>` … `<extra_id_99>`
- Train on the **pre-training corpus** (`whole_func_string` from CodeSearchNet Java)

In [4]:
# ── 1.1  Load 50K Java methods from CodeSearchNet ─────────────────────────────
print('Loading CodeSearchNet Java...')
csn = load_dataset('code_search_net', 'java')
methods_raw = csn['train'].shuffle(seed=SEED).select(range(50_000))

# Extract method bodies
java_methods = [sample['whole_func_string'] for sample in methods_raw]

print(f'Loaded {len(java_methods)} Java methods')
print('\nSample method:')
print(java_methods[0][:300])

Loading CodeSearchNet Java...
Loaded 50000 Java methods

Sample method:
public void put(String entityTypeId, Entity entity) {
    CombinedEntityCache entityCache = caches.get();
    if (entityCache != null) {
      entityCache.put(entity);
      LOG.trace(
          "Added dehydrated row [{}] from entity {} to the L1 cache",
          entity.getIdValue(),
          enti


In [5]:
# ── 1.2  Write corpus to a plain-text file for SentencePiece ──────────────────
# SentencePiece trainer expects a text file with one document per line

CORPUS_FILE = f'{BASE_DIR}/pretrain_corpus.txt'

with open(CORPUS_FILE, 'w', encoding='utf-8') as f:
    for method in java_methods:
        # Replace newlines within a method so each method is one line
        f.write(method.replace('\n', ' ') + '\n')

print(f'Corpus written to {CORPUS_FILE}')

Corpus written to data//pretrain_corpus.txt


In [6]:
# ── 1.3  Train the SentencePiece tokenizer ────────────────────────────────────

SP_MODEL_PREFIX = f'{TOKENIZER_DIR}/spm_java_16k'

# Build the sentinel token string for user_defined_symbols
sentinel_tokens = [f'<extra_id_{i}>' for i in range(100)]
user_symbols = ','.join(sentinel_tokens)

# Retrain — put sentinels at the TOP of vocab using control_symbols
sentinel_tokens_str = ','.join([f'<extra_id_{i}>' for i in range(100)])

spm.SentencePieceTrainer.train(
    input=CORPUS_FILE,
    model_prefix=SP_MODEL_PREFIX,
    model_type='unigram',
    vocab_size=16384,
    character_coverage=0.9999,
    pad_id=0,   pad_piece='<pad>',
    unk_id=2,   unk_piece='<unk>',
    bos_id=-1,
    eos_id=1,   eos_piece='</s>',
    control_symbols=sentinel_tokens_str,
)

print(f'Tokenizer saved: {SP_MODEL_PREFIX}.model')

Tokenizer saved: data//tokenizer/spm_java_16k.model


In [7]:
# ── 1.4  Wrap as a HuggingFace tokenizer & run sanity checks ──────────────────

from transformers import T5Tokenizer

# Load the tokenizer
tokenizer = T5Tokenizer(
    vocab_file=f'{SP_MODEL_PREFIX}.model',
    extra_ids=0,
)
sentinel_list = [f'<extra_id_{i}>' for i in range(100)]
tokenizer.add_special_tokens({'additional_special_tokens': sentinel_list})
tokenizer.save_pretrained(TOKENIZER_DIR)

sentinel_base = tokenizer.convert_tokens_to_ids('<extra_id_0>')
sentinel_99   = tokenizer.convert_tokens_to_ids('<extra_id_99>')
print(f'extra_id_0  : {sentinel_base}')
print(f'extra_id_99 : {sentinel_99}')
print(f'sentinel_base - 99 = {sentinel_base - 99}')

# Save
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f'Vocab size: {len(tokenizer)}')
print(f'Special tokens: {tokenizer.all_special_tokens[:5]}')
print(f'pad_token_id : {tokenizer.pad_token_id}')
print(f'eos_token_id : {tokenizer.eos_token_id}')
print(f'bos_token_id : {tokenizer.bos_token_id}')

# # Check sentinel tokens are single tokens (not split)
for i in [0, 1, 50, 99]:
    tok = f'<extra_id_{i}>'
    ids = tokenizer.encode(tok, add_special_tokens=False)
    print(f'{tok} -> {ids}  (should be exactly 1 token)')
    assert len(ids) == 1, f'Sentinel {tok} was split!'

# # Check a normal Java snippet tokenizes sensibly
sample = 'public void sort(int[] arr) { Arrays.sort(arr); }'
print(tokenizer.tokenize(sample))

print('Tokenizer checks done.')

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


extra_id_0  : 3
extra_id_99 : 102
sentinel_base - 99 = -96
Vocab size: 16384
Special tokens: ['</s>', '<unk>', '<pad>', '<extra_id_0>', '<extra_id_1>']
pad_token_id : 0
eos_token_id : 1
bos_token_id : None
<extra_id_0> -> [3]  (should be exactly 1 token)
<extra_id_1> -> [4]  (should be exactly 1 token)
<extra_id_50> -> [53]  (should be exactly 1 token)
<extra_id_99> -> [102]  (should be exactly 1 token)
['▁public', '▁void', '▁sort', '(', 'int', '[]', '▁arr', ')', '▁{', '▁Arrays', '.', 'sort', '(', 'arr', ');', '▁}']
Tokenizer checks done.


---
# Section 2 — Pre-training (Pipeline A)

**Goal:** Pre-train T5-small from scratch using span corruption on the 50K Java corpus.  

**Requirements:**
- T5Config defined **manually** (not `AutoConfig`)
- Corruption rate: 15%, using sentinel tokens
- Token-length filter: keep methods with 10–512 tokens
- Training: flat 3 epochs, no validation split, no early stopping
- Log training loss per epoch (must be decreasing)

In [8]:
# ── 2.1  Define T5Config manually ─────────────────────────────────────────────
#
# This produces T5-small (~60M params): 6 enc layers, 6 dec layers,
# d_model=512, d_ff=2048, d_kv=64, 8 attention heads
#
t5_config = T5Config(
    vocab_size=len(tokenizer),
    d_model=512,
    d_ff=2048,
    d_kv=64,
    num_heads=8,
    num_layers=6,
    num_decoder_layers=6,
    decoder_start_token_id=0, 
    eos_token_id=1,   
    bos_token_id=0,            
    pad_token_id=0,  
)

model_pt = T5ForConditionalGeneration(config=t5_config).to(DEVICE)
model_pt.resize_token_embeddings(len(tokenizer))

total_params = sum(p.numel() for p in model_pt.parameters()) / 1e6
print(f'Model parameters: {total_params:.1f}M  (expect ~60M)')

Model parameters: 52.4M  (expect ~60M)


In [9]:
# ── 2.2  Filter corpus: keep methods with 10–512 tokens ───────────────────────

filtered_methods = []
for method in tqdm(java_methods, desc='Filtering'):
    toks = tokenizer.encode(method, truncation=False)
    if 10 <= len(toks) <= 512:
        filtered_methods.append(method)

print(f'After filtering: {len(filtered_methods)} / {len(java_methods)} methods kept')

Filtering: 100%|███████████████████████████████████████████████████████████████| 50000/50000 [00:34<00:00, 1459.12it/s]

After filtering: 48063 / 50000 methods kept


In [10]:
# ── 2.3  Span Corruption ───────────────────────────────────────────────────────
#
# Adapted from the lab notebook. This version uses SentencePiece tokenizer instead of the codet5p tokenizer
#
# How it works:
#   1. Randomly select ~15% of eligible (non-special) token positions
#   2. Group consecutive selected positions into spans
#   3. Replace each span with one sentinel: <extra_id_0>, <extra_id_1>, ...
#   4. Decoder target = sentinel + dropped tokens, for each span, + final sentinel

def span_corruption(input_ids: torch.Tensor, tokenizer, corruption_rate=0.15):
    input_ids = input_ids.squeeze()
    length = input_ids.size(0)

    special_ids = set(filter(None, [
        tokenizer.pad_token_id,
        tokenizer.eos_token_id,
        tokenizer.bos_token_id,
    ]))

    eligible = [i for i in range(length) if input_ids[i].item() not in special_ids]

    if not eligible:
        sentinel_id = tokenizer.convert_tokens_to_ids('<extra_id_0>')
        return input_ids, torch.tensor([sentinel_id], dtype=torch.long)

    num_to_corrupt = max(1, int(len(eligible) * corruption_rate))
    corrupted_positions = set(sorted(random.sample(eligible, min(num_to_corrupt, len(eligible)))))

    spans, current_span = [], []
    for pos in sorted(corrupted_positions):
        if current_span and pos != current_span[-1] + 1:
            spans.append(current_span)
            current_span = []
        current_span.append(pos)
    if current_span:
        spans.append(current_span)

    # Get ALL sentinel IDs explicitly by name — don't do math
    sentinel_ids = [
        tokenizer.convert_tokens_to_ids(f'<extra_id_{i}>')
        for i in range(len(spans) + 1)   # +1 for the final closing sentinel
    ]

    # Build encoder input
    encoder_tokens = []
    span_idx, i = 0, 0
    while i < length:
        if span_idx < len(spans) and i == spans[span_idx][0]:
            encoder_tokens.append(sentinel_ids[span_idx])   # <-- no math, direct lookup
            i = spans[span_idx][-1] + 1
            span_idx += 1
        else:
            encoder_tokens.append(input_ids[i].item())
            i += 1

    # Build decoder target
    decoder_tokens = []
    for idx, span in enumerate(spans):
        decoder_tokens.append(sentinel_ids[idx])             # <-- direct lookup
        decoder_tokens.extend(input_ids[pos].item() for pos in span)
    decoder_tokens.append(sentinel_ids[len(spans)])          # closing sentinel

    return (
        torch.tensor(encoder_tokens, dtype=torch.long),
        torch.tensor(decoder_tokens, dtype=torch.long),
    )
print('span_corruption() defined.')

span_corruption() defined.


In [11]:
# ── 2.4  Dataset & DataLoader ──────────────────────────────────────────────────

class SpanCorruptionDataset(Dataset):
    def __init__(self, code_snippets, tokenizer, max_length=512, corruption_rate=0.15):
        self.tokenizer = tokenizer
        self.corruption_rate = corruption_rate
        self.token_ids = [
            tokenizer.encode(code, max_length=max_length, truncation=True)
            for code in tqdm(code_snippets, desc='Tokenizing corpus')
        ]

    def __len__(self):
        return len(self.token_ids)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.token_ids[idx], dtype=torch.long)
        enc_input, dec_target = span_corruption(input_ids, self.tokenizer, self.corruption_rate)
        return {'input_ids': enc_input, 'labels': dec_target}


def pretrain_collate_fn(batch):
    """Pad variable-length encoder inputs and decoder targets within each batch."""
    input_ids_list = [item['input_ids'] for item in batch]
    labels_list    = [item['labels']    for item in batch]

    max_in  = max(x.size(0) for x in input_ids_list)
    max_lbl = max(x.size(0) for x in labels_list)

    padded_inputs  = torch.full((len(batch), max_in),  tokenizer.pad_token_id, dtype=torch.long)
    attn_masks     = torch.zeros((len(batch), max_in),  dtype=torch.long)
    padded_labels  = torch.full((len(batch), max_lbl), -100, dtype=torch.long)

    for i, (inp, lbl) in enumerate(zip(input_ids_list, labels_list)):
        padded_inputs[i, :inp.size(0)] = inp
        attn_masks[i,    :inp.size(0)] = 1
        padded_labels[i, :lbl.size(0)] = lbl

    return {'input_ids': padded_inputs, 'attention_mask': attn_masks, 'labels': padded_labels}


# Create dataset — NO validation split (assignment requirement)
PRETRAIN_BATCH_SIZE = 16 

pretrain_dataset = SpanCorruptionDataset(filtered_methods, tokenizer, max_length=512)
pretrain_loader  = DataLoader(pretrain_dataset, batch_size=PRETRAIN_BATCH_SIZE,
                               shuffle=True, collate_fn=pretrain_collate_fn)

print(f'Pre-training batches: {len(pretrain_loader)}')

Tokenizing corpus: 100%|███████████████████████████████████████████████████████| 48063/48063 [00:27<00:00, 1773.07it/s]

Pre-training batches: 3004


In [12]:
sentinel_base = tokenizer.convert_tokens_to_ids('<extra_id_0>')
print(f'sentinel_base      : {sentinel_base}')
print(f'vocab size         : {len(tokenizer)}')
print(f'sentinel_base < vocab_size: {sentinel_base < len(tokenizer)}')

# Also check the last sentinel
sentinel_99 = tokenizer.convert_tokens_to_ids('<extra_id_99>')
print(f'extra_id_99 ID     : {sentinel_99}')
print(f'sentinel_base - 99 : {sentinel_base - 99}')  # should be > 0

sentinel_base      : 3
vocab size         : 16384
sentinel_base < vocab_size: True
extra_id_99 ID     : 102
sentinel_base - 99 : -96


In [13]:
# ── 2.5  Pre-training Loop (3 epochs, flat — no early stopping) ───────────────
PRETRAIN_EPOCHS = 3
PRETRAIN_LR     = 1e-4 

optimizer_pt = torch.optim.AdamW(model_pt.parameters(), lr=PRETRAIN_LR)

pretrain_epoch_losses = []

for epoch in range(PRETRAIN_EPOCHS):
    model_pt.train()
    epoch_loss, n_batches = 0.0, 0

    progress = tqdm(pretrain_loader, desc=f'Pre-train Epoch {epoch+1}/{PRETRAIN_EPOCHS}')
    for batch in progress:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        outputs = model_pt(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer_pt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pt.parameters(), max_norm=1.0)
        optimizer_pt.step()

        epoch_loss += loss.item()
        n_batches  += 1
        progress.set_postfix(loss=f'{loss.item():.4f}')

    avg = epoch_loss / n_batches
    pretrain_epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/{PRETRAIN_EPOCHS} | Avg Train Loss: {avg:.4f}')

print(f'\nPre-training complete. Losses per epoch: {pretrain_epoch_losses}')
print('Loss should be decreasing across epochs.')

# Save
model_pt.save_pretrained(PRETRAIN_DIR)
tokenizer.save_pretrained(PRETRAIN_DIR)
print(f'Pre-trained model saved to {PRETRAIN_DIR}')

Pre-train Epoch 1/3: 100%|████████████████████████████████████████████| 3004/3004 [34:41<00:00,  1.44it/s, loss=3.5915]


Epoch 1/3 | Avg Train Loss: 4.1720


Pre-train Epoch 2/3: 100%|████████████████████████████████████████████| 3004/3004 [33:48<00:00,  1.48it/s, loss=3.7014]


Epoch 2/3 | Avg Train Loss: 3.7040


Pre-train Epoch 3/3: 100%|████████████████████████████████████████████| 3004/3004 [35:10<00:00,  1.42it/s, loss=3.6975]


Epoch 3/3 | Avg Train Loss: 3.6251

Pre-training complete. Losses per epoch: [4.171960439647085, 3.703999168625208, 3.6251311811721436]
Loss should be decreasing across epochs.
Pre-trained model saved to data//model_pretrained


---
# Section 3 — Fine-tuning: Pipeline A (Pre-trained -> Bug Fixing) 

**Dataset:** `google/code_x_glue_cc_code_refinement` (name=`'medium'`)  
- Input: `buggy` field  
- Output: `fixed` field  
- ~52K train / ~6.5K val / ~6.5K test

Use the **same hyperparameters** as Pipeline B so the only variable is pre-training.

In [14]:
# ── 3.1  Load bug-fixing dataset ──────────────────────────────────────────────

bugfix_ds = load_dataset('google/code_x_glue_cc_code_refinement', name='medium')

print(bugfix_ds)
print('\nSample:')
print('buggy :', bugfix_ds['train'][0]['buggy'][:200])
print('fixed :', bugfix_ds['train'][0]['fixed'][:200])

DatasetDict({
    train: Dataset({
        features: ['id', 'buggy', 'fixed'],
        num_rows: 52364
    })
    validation: Dataset({
        features: ['id', 'buggy', 'fixed'],
        num_rows: 6546
    })
    test: Dataset({
        features: ['id', 'buggy', 'fixed'],
        num_rows: 6545
    })
})

Sample:
buggy : public static TYPE_1 init ( java.lang.String name , java.util.Date date ) { TYPE_1 VAR_1 = new TYPE_1 ( ) ; VAR_1 . METHOD_1 ( name ) ; java.util.Calendar VAR_2 = java.util.Calendar.getInstance ( ) ; 
fixed : public static TYPE_1 init ( java.lang.String name , java.util.Date date ) { TYPE_1 VAR_1 = new TYPE_1 ( ) ; VAR_1 . METHOD_1 ( name ) ; java.util.Calendar VAR_2 = null ; if ( date != null ) { VAR_2 = 


In [15]:
# ── 3.2  Bug-fixing Dataset class ─────────────────────────────────────────────

class BugFixDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_src_len=512, max_tgt_len=512):
        self.tokenizer   = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        self.samples = []
        for item in tqdm(hf_split, desc='Tokenizing bug-fix pairs'):
            src = tokenizer.encode(item['buggy'], max_length=max_src_len, truncation=True)
            tgt = tokenizer.encode(item['fixed'], max_length=max_tgt_len, truncation=True)
            self.samples.append({'input_ids': src, 'labels': tgt})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'labels':    torch.tensor(item['labels'],    dtype=torch.long),
        }


def finetune_collate_fn(batch):
    input_ids_list = [item['input_ids'] for item in batch]
    labels_list    = [item['labels']    for item in batch]

    max_in  = max(x.size(0) for x in input_ids_list)
    max_lbl = max(x.size(0) for x in labels_list)

    padded_inputs = torch.full((len(batch), max_in),  tokenizer.pad_token_id, dtype=torch.long)
    attn_masks    = torch.zeros((len(batch), max_in),  dtype=torch.long)
    padded_labels = torch.full((len(batch), max_lbl), -100, dtype=torch.long)

    for i, (inp, lbl) in enumerate(zip(input_ids_list, labels_list)):
        padded_inputs[i, :inp.size(0)] = inp
        attn_masks[i,    :inp.size(0)] = 1
        padded_labels[i, :lbl.size(0)] = lbl

    return {'input_ids': padded_inputs, 'attention_mask': attn_masks, 'labels': padded_labels}


# ── Shared fine-tuning hyperparameters (used for BOTH Pipeline A and B) ────────
FT_BATCH_SIZE = 16  
FT_EPOCHS     = 5 
FT_LR         = 5e-5 
MAX_SRC_LEN   = 512
MAX_TGT_LEN   = 512

train_ft_dataset = BugFixDataset(bugfix_ds['train'],      tokenizer, MAX_SRC_LEN, MAX_TGT_LEN)
val_ft_dataset   = BugFixDataset(bugfix_ds['validation'], tokenizer, MAX_SRC_LEN, MAX_TGT_LEN)
test_ft_dataset  = BugFixDataset(bugfix_ds['test'],       tokenizer, MAX_SRC_LEN, MAX_TGT_LEN)

train_ft_loader = DataLoader(train_ft_dataset, batch_size=FT_BATCH_SIZE, shuffle=True,  collate_fn=finetune_collate_fn)
val_ft_loader   = DataLoader(val_ft_dataset,   batch_size=FT_BATCH_SIZE, shuffle=False, collate_fn=finetune_collate_fn)

print(f'Train: {len(train_ft_dataset)} | Val: {len(val_ft_dataset)} | Test: {len(test_ft_dataset)}')

Tokenizing bug-fix pairs: 100%|███████████████████████████████████████████████████| 6545/6545 [00:07<00:00, 883.08it/s]

Train: 52364 | Val: 6546 | Test: 6545


In [16]:
# ── 3.3  Fine-tuning helper functions ─────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, n = 0.0, 0
    for batch in tqdm(loader, desc='  train', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n += 1
    return total_loss / n


def evaluate_loss(model, loader, device):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(loader, desc='  val', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
            total_loss += loss.item()
            n += 1
    return total_loss / n

print('Helper functions defined.')

Helper functions defined.


In [17]:
# ── 3.4  Fine-tune Pipeline A (load pre-trained model) ────────────────────────

print('Loading pre-trained model for Pipeline A...')
model_A = T5ForConditionalGeneration.from_pretrained(PRETRAIN_DIR).to(DEVICE)

optimizer_A = torch.optim.AdamW(model_A.parameters(), lr=FT_LR)

ft_A_train_losses, ft_A_val_losses = [], []

for epoch in range(FT_EPOCHS):
    train_loss = train_one_epoch(model_A, train_ft_loader, optimizer_A, DEVICE)
    val_loss   = evaluate_loss(model_A, val_ft_loader, DEVICE)
    ft_A_train_losses.append(train_loss)
    ft_A_val_losses.append(val_loss)
    print(f'[Pipeline A] Epoch {epoch+1}/{FT_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

model_A.save_pretrained(FINETUNE_A_DIR)
print(f'Pipeline A model saved to {FINETUNE_A_DIR}')

Loading pre-trained model for Pipeline A...


[Pipeline A] Epoch 1/5 | Train: 1.0278 | Val: 0.6349


[Pipeline A] Epoch 2/5 | Train: 0.6096 | Val: 0.4784


[Pipeline A] Epoch 3/5 | Train: 0.4874 | Val: 0.4108


[Pipeline A] Epoch 4/5 | Train: 0.4302 | Val: 0.3709


[Pipeline A] Epoch 5/5 | Train: 0.3970 | Val: 0.3472
Pipeline A model saved to data//model_finetuned_A


---
# Section 4 — Fine-tuning: Pipeline B (Scratch -> Bug Fixing)

**Goal:** Initialize a fresh T5-small (same config, same tokenizer) and fine-tune directly.  
No pre-training weights, this model starts with random weights.

**Must use identical hyperparameters to Pipeline A.**

In [18]:
# ── 4.1  Initialize Pipeline B model from scratch ─────────────────────────────
#
# Same T5Config as Section 2.1

model_B = T5ForConditionalGeneration(config=t5_config).to(DEVICE)
model_B.resize_token_embeddings(len(tokenizer))

params_B = sum(p.numel() for p in model_B.parameters()) / 1e6
print(f'Pipeline B model: {params_B:.1f}M params (randomly initialized)')

optimizer_B = torch.optim.AdamW(model_B.parameters(), lr=FT_LR)  # same LR as A

ft_B_train_losses, ft_B_val_losses = [], []

for epoch in range(FT_EPOCHS):  # same FT_EPOCHS as A
    train_loss = train_one_epoch(model_B, train_ft_loader, optimizer_B, DEVICE)
    val_loss   = evaluate_loss(model_B, val_ft_loader, DEVICE)
    ft_B_train_losses.append(train_loss)
    ft_B_val_losses.append(val_loss)
    print(f'[Pipeline B] Epoch {epoch+1}/{FT_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

model_B.save_pretrained(FINETUNE_B_DIR)
print(f'Pipeline B model saved to {FINETUNE_B_DIR}')

Pipeline B model: 52.4M params (randomly initialized)


[Pipeline B] Epoch 1/5 | Train: 1.0480 | Val: 0.5546


[Pipeline B] Epoch 2/5 | Train: 0.5297 | Val: 0.4139


[Pipeline B] Epoch 3/5 | Train: 0.4294 | Val: 0.3625


[Pipeline B] Epoch 4/5 | Train: 0.3888 | Val: 0.3413


[Pipeline B] Epoch 5/5 | Train: 0.3633 | Val: 0.3241
Pipeline B model saved to data//model_finetuned_B


---
# Section 5 — RAG: Knowledge Base & Retriever

**Goal:** Build a retrieval system over the bug-fixing training set.  
- **Knowledge base:** buggy -> fixed pairs from `code_x_glue_cc_code_refinement` train split
- **Generator:** `Qwen2.5-Coder-1.5B-Instruct`

In [19]:
import faiss
from gensim.models import Word2Vec

print('RAG imports done.')

RAG imports done.


In [20]:
# ── 5.1  Build knowledge base from bug-fixing training data ───────────────────
#
# Each KB entry should store:
#   - 'buggy'   : the buggy code snippet (used as the retrieval key)
#   - 'fixed'   : the corresponding fix (shown in few-shot context)
#
# We embed the 'buggy' field and store it in FAISS so that at inference
# time we can retrieve similar buggy examples given a new buggy query.

kb_train = bugfix_ds['train']  # already loaded above

kb_entries = [
    {'buggy': item['buggy'], 'fixed': item['fixed']}
    for item in kb_train
]

print(f'Knowledge base size: {len(kb_entries)} bug-fix pairs')

Knowledge base size: 52364 bug-fix pairs


In [21]:
# ── 5.2  Train retrieval embeddings ───────────────────────────────────────────

def tokenize_code(text):
    """Simple whitespace tokenizer."""
    return text.lower().split()


class CodeEmbedder:
    """Word2Vec skip-gram + mean pooling."""

    def __init__(self, vector_size=128, window=5, min_count=2, workers=4):
        self.vector_size = vector_size
        self.window      = window
        self.min_count   = min_count
        self.workers     = workers
        self.model       = None

    def train(self, corpus):
        """corpus: list of code strings."""
        tokenized = [tokenize_code(doc) for doc in tqdm(corpus, desc='Tokenizing for W2V')]
        self.model = Word2Vec(
            sentences=tokenized,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            workers=self.workers,
            sg=1,   # skip-gram
            epochs=10,
        )
        print(f'W2V vocab: {len(self.model.wv)} | dim: {self.vector_size}')

    def encode(self, texts):
        embeddings = []
        for text in texts:
            tokens = tokenize_code(text)
            vecs   = [self.model.wv[t] for t in tokens if t in self.model.wv]
            embeddings.append(np.mean(vecs, axis=0) if vecs else np.zeros(self.vector_size))
        return np.array(embeddings, dtype=np.float32)


# Train on buggy snippets (the retrieval key)
buggy_corpus = [e['buggy'] for e in kb_entries]

embedder = CodeEmbedder(vector_size=128)
embedder.train(buggy_corpus)

# Embed the entire knowledge base
print('Embedding KB...')
kb_embeddings = embedder.encode(buggy_corpus)
print(f'KB embeddings shape: {kb_embeddings.shape}')

Tokenizing for W2V: 100%|████████████████████████████████████████████████████| 52364/52364 [00:00<00:00, 139246.82it/s]


W2V vocab: 469 | dim: 128
Embedding KB...
KB embeddings shape: (52364, 128)


In [22]:
# ── 5.3  Build FAISS index ─────────────────────────────────────────────────────

dim        = kb_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(kb_embeddings)

print(f'FAISS index: {faiss_index.ntotal} vectors (dim={dim})')

# Save index and metadata for later reuse
faiss.write_index(faiss_index, f'{RAG_KB_DIR}/bugfix_index.faiss')
with open(f'{RAG_KB_DIR}/kb_metadata.pkl', 'wb') as f:
    pickle.dump(kb_entries, f)
embedder.model.save(f'{RAG_KB_DIR}/word2vec.model')

print(f'KB saved to {RAG_KB_DIR}')

FAISS index: 52364 vectors (dim=128)
KB saved to data//rag_kb


In [23]:
# ── 5.4  Retriever class ───────────────────────────────────────────────────────

class BugFixRetriever:
    def __init__(self, faiss_index, kb_entries, embedder):
        self.index     = faiss_index
        self.kb        = kb_entries
        self.embedder  = embedder

    def retrieve(self, query: str, top_k: int = 3):
        query_vec              = self.embedder.encode([query])  # shape (1, dim)
        distances, indices     = self.index.search(query_vec, top_k)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx < len(self.kb):
                entry = self.kb[idx].copy()
                entry['similarity'] = float(1 / (1 + dist))
                results.append(entry)
        return results


retriever = BugFixRetriever(faiss_index, kb_entries, embedder)

# Quick sanity check
sample_query   = bugfix_ds['test'][0]['buggy']
sample_results = retriever.retrieve(sample_query, top_k=3)
print(f'Query (first 100 chars): {sample_query[:100]}')
for i, r in enumerate(sample_results, 1):
    print(f'  {i}. similarity={r["similarity"]:.4f} | buggy[:60]: {r["buggy"][:60]}')

Query (first 100 chars): public java.lang.String METHOD_1 ( ) { if ( ( METHOD_2 ( ) ) && ( METHOD_3 ( VAR_1 . METHOD_4 ( ) ) 
  1. similarity=0.9866 | buggy[:60]: private TYPE_1 METHOD_1 ( ) { if ( VAR_1 . METHOD_2 ( ) ) { 
  2. similarity=0.9822 | buggy[:60]: public void METHOD_1 ( ) { METHOD_2 ( ) ; if ( VAR_1 . METHO
  3. similarity=0.9809 | buggy[:60]: public void METHOD_1 ( ) { if ( VAR_1 ) { VAR_2 . METHOD_2 (


---
# Section 6 — RAG Inference

**Goal:** Generate bug fixes using Qwen2.5-Coder-1.5B-Instruct in two settings:  
1. **With RAG** — retrieve ≥3 similar examples and include them as few-shot context  
2. **Zero-shot** — provide only the buggy method, no retrieved examples  

Both settings are evaluated on the **same test set** used for Pipelines A and B.

In [24]:
# ── 6.1  Load Qwen2.5-Coder-1.5B-Instruct ────────────────────────────────────

QWEN_MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

print(f'Loading {QWEN_MODEL_ID}...')
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
qwen_model     = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.float16,
    device_map='cuda:0',
)
qwen_model.eval()
print('Qwen loaded.')

Loading Qwen/Qwen2.5-Coder-1.5B-Instruct...
Qwen loaded.


In [25]:
# ── 6.2  Prompt builders ──────────────────────────────────────────────────────

def build_few_shot_prompt(buggy_code: str, retrieved_examples: list) -> str:
    """
    Build a few-shot prompt with retrieved bug-fix examples as context.
    Use at least 3 retrieved examples (assignment requirement).
    """
    prompt = 'Fix the bug in the following Java method.\n\n'
    prompt += 'Here are similar bug fixes for reference:\n\n'

    for i, ex in enumerate(retrieved_examples, 1):
        prompt += f'### Example {i}\n'
        prompt += f'Buggy:\n{ex["buggy"]}\n\n'
        prompt += f'Fixed:\n{ex["fixed"]}\n\n'

    prompt += '### Now fix this:\n'
    prompt += f'Buggy:\n{buggy_code}\n\n'
    prompt += 'Fixed:\n'
    return prompt


def build_zero_shot_prompt(buggy_code: str) -> str:
    """
    Zero-shot prompt — no retrieved examples.
    """
    return (
        'Fix the bug in the following Java method. '
        'Return only the corrected method, no explanation.\n\n'
        f'{buggy_code}\n\nFixed:\n'
    )


def generate_batch_qwen(prompts: list, max_new_tokens: int = 128) -> list:
    inputs = qwen_tokenizer(
        prompts,
        return_tensors='pt',
        truncation=True,
        max_length=2048,
        padding=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    results = []
    for i, out in enumerate(output_ids):
        new_tokens = out[inputs['input_ids'].shape[1]:]
        results.append(qwen_tokenizer.decode(new_tokens, skip_special_tokens=True))
    return results

print('Prompt builders and generator defined.')

Prompt builders and generator defined.


In [26]:
print(f'DEVICE: {DEVICE}')
print(f'Model device: {next(qwen_model.parameters()).device}')

# Test with a single simple prompt before the loop
test_prompt = "Fix this: public int add(int a, int b) { return a - b; }"
inputs = qwen_tokenizer(test_prompt, return_tensors='pt', truncation=True, max_length=128)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
print(f'Input device: {inputs["input_ids"].device}')

DEVICE: cuda
Model device: cuda:0
Input device: cuda:0


In [27]:
# ── 6.3  Run RAG inference on the test set ────────────────────────────────────
#
# For each test sample:
#   1. Retrieve top-K similar bug-fix examples from the KB
#   2. Build few-shot prompt
#   3. Generate with Qwen
#   4. Store prediction

TOP_K = 3
test_samples = list(bugfix_ds['test'])

rag_predictions  = []   # generated fixes (with RAG)
rag_references   = []   # ground-truth fixes

BATCH_SIZE = 32
for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc='RAG inference'):
    batch = test_samples[i:i+BATCH_SIZE]
    prompts = [build_few_shot_prompt(s['buggy'], retriever.retrieve(s['buggy'], top_k=TOP_K)) for s in batch]
    predictions = generate_batch_qwen(prompts)
    rag_predictions.extend([p.strip() for p in predictions])
    rag_references.extend([s['fixed'].strip() for s in batch])
# Save
with open(f'{RESULTS_DIR}/rag_predictions.json', 'w') as f:
    json.dump({'predictions': rag_predictions, 'references': rag_references}, f)

print(f'RAG inference done. {len(rag_predictions)} predictions saved.')

RAG inference: 100%|█████████████████████████████████████████████████████████████| 205/205 [16:34:56<00:00, 291.20s/it]


RAG inference done. 6545 predictions saved.


In [32]:
# ── 6.4  Run Zero-shot inference on the test set ──────────────────────────────
zeroshot_predictions = []
zeroshot_references  = []

BATCH_SIZE = 32  # same as RAG section

for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc='Zero-shot inference'):
    batch = test_samples[i:i+BATCH_SIZE]
    prompts = [build_zero_shot_prompt(s['buggy']) for s in batch]
    predictions = generate_batch_qwen(prompts)
    zeroshot_predictions.extend([p.strip() for p in predictions])
    zeroshot_references.extend([s['fixed'].strip() for s in batch])

with open(f'{RESULTS_DIR}/zeroshot_predictions.json', 'w') as f:
    json.dump({'predictions': zeroshot_predictions, 'references': zeroshot_references}, f)

print(f'Zero-shot inference done. {len(zeroshot_predictions)} predictions saved.')

Zero-shot inference: 100%|█████████████████████████████████████████████████████████| 205/205 [4:19:18<00:00, 75.89s/it]


Zero-shot inference done. 6545 predictions saved.


---
# Section 7 — Evaluation

**Metrics (computed on the test set only):**
1. **CodeBLEU** — code-aware metric combining n-gram, syntax, and dataflow scores
2. **Exact Match** — fraction of predictions that exactly match the reference fix

**Evaluated for all 4 configurations:**
| Config | Model | Strategy |
|--------|-------|----------|
| A | T5-small (pre-trained → fine-tuned) | Fine-tuning |
| B | T5-small (scratch → fine-tuned) | Fine-tuning |
| C | Qwen2.5-Coder-1.5B | RAG (few-shot) |
| D | Qwen2.5-Coder-1.5B | Zero-shot |

In [9]:
# Section 0 (re-run as normal first), then run this:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Reload tokenizer
tokenizer = T5Tokenizer.from_pretrained(TOKENIZER_DIR)
print(f'Tokenizer loaded. Vocab size: {len(tokenizer)}')

# Reload RAG predictions
with open(f'{RESULTS_DIR}/rag_predictions.json', 'r') as f:
    rag_data = json.load(f)
    rag_predictions = rag_data['predictions']
    rag_references  = rag_data['references']

# Reload zero-shot predictions
with open(f'{RESULTS_DIR}/zeroshot_predictions.json', 'r') as f:
    zs_data = json.load(f)
    zeroshot_predictions = zs_data['predictions']
    zeroshot_references  = zs_data['references']

# Reload bug-fixing dataset and recreate test split
from datasets import load_dataset
from torch.utils.data import DataLoader


class BugFixDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_src_len=512, max_tgt_len=512):
        self.tokenizer   = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        self.samples = []
        for item in tqdm(hf_split, desc='Tokenizing bug-fix pairs'):
            src = tokenizer.encode(item['buggy'], max_length=max_src_len, truncation=True)
            tgt = tokenizer.encode(item['fixed'], max_length=max_tgt_len, truncation=True)
            self.samples.append({'input_ids': src, 'labels': tgt})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'labels':    torch.tensor(item['labels'],    dtype=torch.long),
        }


def finetune_collate_fn(batch):
    input_ids_list = [item['input_ids'] for item in batch]
    labels_list    = [item['labels']    for item in batch]

    max_in  = max(x.size(0) for x in input_ids_list)
    max_lbl = max(x.size(0) for x in labels_list)

    padded_inputs = torch.full((len(batch), max_in),  tokenizer.pad_token_id, dtype=torch.long)
    attn_masks    = torch.zeros((len(batch), max_in),  dtype=torch.long)
    padded_labels = torch.full((len(batch), max_lbl), -100, dtype=torch.long)

    for i, (inp, lbl) in enumerate(zip(input_ids_list, labels_list)):
        padded_inputs[i, :inp.size(0)] = inp
        attn_masks[i,    :inp.size(0)] = 1
        padded_labels[i, :lbl.size(0)] = lbl

    return {'input_ids': padded_inputs, 'attention_mask': attn_masks, 'labels': padded_labels}



bugfix_ds = load_dataset('google/code_x_glue_cc_code_refinement', name='medium')
test_ft_dataset = BugFixDataset(bugfix_ds['test'], tokenizer, max_src_len=512, max_tgt_len=512)
print(f'Test dataset loaded: {len(test_ft_dataset)} samples')

print(f'RAG preds: {len(rag_predictions)} | Zero-shot preds: {len(zeroshot_predictions)}')

Tokenizer loaded. Vocab size: 16284


Tokenizing bug-fix pairs: 100%|██████████████████████████████████████████████████| 6545/6545 [00:06<00:00, 1011.63it/s]

Test dataset loaded: 6545 samples
RAG preds: 6545 | Zero-shot preds: 6545


In [14]:
# ── 7.1  Generate predictions from fine-tuned T5 models ───────────────────────
#
# Run beam search / greedy decode on the test set for both Pipeline A and B.

NUM_BEAMS   = 2
MAX_GEN_LEN = 256

def generate_t5_predictions(model, tokenizer, test_dataset, device, num_beams=4, max_len=512):
    model.eval()
    predictions, references = [], []

    for sample in tqdm(test_dataset.samples, desc='Generating'):
        input_ids      = torch.tensor(sample['input_ids']).unsqueeze(0).to(device)
        attention_mask = (input_ids != tokenizer.pad_token_id).long()

        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=max_len,
                num_beams=num_beams,
                early_stopping=True,
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        ref  = tokenizer.decode(
            [t for t in sample['labels'] if t != -100],
            skip_special_tokens=True
        )
        predictions.append(pred.strip())
        references.append(ref.strip())

    return predictions, references


print('Loading fine-tuned models...')
model_A_eval = T5ForConditionalGeneration.from_pretrained(FINETUNE_A_DIR).to(DEVICE)
model_B_eval = T5ForConditionalGeneration.from_pretrained(FINETUNE_B_DIR).to(DEVICE)

print('Generating Pipeline A predictions...')
preds_A, refs_A = generate_t5_predictions(model_A_eval, tokenizer, test_ft_dataset, DEVICE, NUM_BEAMS, MAX_GEN_LEN)

print('Generating Pipeline B predictions...')
preds_B, refs_B = generate_t5_predictions(model_B_eval, tokenizer, test_ft_dataset, DEVICE, NUM_BEAMS, MAX_GEN_LEN)

# Save
for name, preds, refs in [('pipeline_A', preds_A, refs_A), ('pipeline_B', preds_B, refs_B)]:
    with open(f'{RESULTS_DIR}/{name}_predictions.json', 'w') as f:
        json.dump({'predictions': preds, 'references': refs}, f)

print('T5 predictions saved.')

Loading fine-tuned models...
Generating Pipeline A predictions...


Generating: 100%|████████████████████████████████████████████████████████████████| 6545/6545 [6:55:15<00:00,  3.81s/it]


Generating Pipeline B predictions...


Generating: 100%|████████████████████████████████████████████████████████████████| 6545/6545 [6:04:49<00:00,  3.34s/it]


T5 predictions saved.


In [15]:
# ── 7.2  Compute CodeBLEU and Exact Match ─────────────────────────────────────

from codebleu import calc_codebleu

def compute_exact_match(predictions, references):
    """Fraction of predictions that exactly match the reference."""
    assert len(predictions) == len(references)
    matches = sum(p.strip() == r.strip() for p, r in zip(predictions, references))
    return matches / len(references)


def compute_codebleu(predictions, references, lang='java'):
    """
    Compute CodeBLEU using the codebleu package.
    """
    refs_wrapped = [[r] for r in references]
    result = calc_codebleu(refs_wrapped, predictions, lang=lang)
    return result


print('Evaluating all configurations...')

configs = {
    'Pipeline A (pre-train + fine-tune)' : (preds_A,            refs_A),
    'Pipeline B (fine-tune only)'        : (preds_B,            refs_B),
    'RAG (Qwen few-shot)'                : (rag_predictions,    rag_references),
    'Zero-shot (Qwen)'                   : (zeroshot_predictions, zeroshot_references),
}

results_table = {}
for name, (preds, refs) in configs.items():
    em       = compute_exact_match(preds, refs)
    cb       = compute_codebleu(preds, refs, lang='java')
    results_table[name] = {
        'exact_match' : em,
        'codebleu'    : cb,
    }
    print(f'\n{name}')
    print(f'  Exact Match : {em:.4f}')
    print(f'  CodeBLEU    : {cb}')

# Save full results
with open(f'{RESULTS_DIR}/all_results.json', 'w') as f:
    json.dump(results_table, f, indent=2)

print(f'\nResults saved to {RESULTS_DIR}/all_results.json')

Evaluating all configurations...

Pipeline A (pre-train + fine-tune)
  Exact Match : 0.0005
  CodeBLEU    : {'codebleu': 0.46258688067480896, 'ngram_match_score': 0.46443056939307964, 'weighted_ngram_match_score': 0.46780724739777596, 'syntax_match_score': 0.46617774623452407, 'dataflow_match_score': 0.4519319596738563}

Pipeline B (fine-tune only)
  Exact Match : 0.0003
  CodeBLEU    : {'codebleu': 0.46440922867926854, 'ngram_match_score': 0.4549783884148039, 'weighted_ngram_match_score': 0.4623605896478245, 'syntax_match_score': 0.47570380884788266, 'dataflow_match_score': 0.46459412780656306}

RAG (Qwen few-shot)
  Exact Match : 0.0000
  CodeBLEU    : {'codebleu': 0.3482041458047155, 'ngram_match_score': 0.15924218637064913, 'weighted_ngram_match_score': 0.17417355941264595, 'syntax_match_score': 0.5227900007885813, 'dataflow_match_score': 0.5366108366469856}

Zero-shot (Qwen)
  Exact Match : 0.0000
  CodeBLEU    : {'codebleu': 0.5524778737064844, 'ngram_match_score': 0.381114570411

---
# Section 8 — Results Summary & Analysis

In [16]:
# ── 8.1  Print final results table ────────────────────────────────────────────

print(f'{"Configuration":<40} {"Exact Match":>12} {"CodeBLEU":>12}')
print('-' * 66)
for name, metrics in results_table.items():
    em = metrics['exact_match']
    
    cb_score = metrics['codebleu']
    if isinstance(cb_score, dict):
        cb_score = cb_score.get('codebleu', cb_score)
    print(f'{name:<40} {em:>12.4f} {cb_score:>12.4f}')

Configuration                             Exact Match     CodeBLEU
------------------------------------------------------------------
Pipeline A (pre-train + fine-tune)             0.0005       0.4626
Pipeline B (fine-tune only)                    0.0003       0.4644
RAG (Qwen few-shot)                            0.0000       0.3482
Zero-shot (Qwen)                               0.0000       0.5525
